In [1]:
from custom_ddim_scheduler import CustomDDIMScheduler
from main import generate_batch
from diffusers import UNet2DModel
from rewards import reward_function
import torch

/home/amanda_mendes_cloudwalk_io/Fine-Tuning-Diffusion-Models-With-RL/.env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'src'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the model and scheduler
scheduler = CustomDDIMScheduler.from_pretrained("google/ddpm-celebahq-256", use_safetensors = True)
pretrained_model = UNet2DModel.from_pretrained("google/ddpm-celebahq-256").to(device)

# Set the timesteps and move the alphas_cumprod to the device
scheduler.set_timesteps(50, device=device)
scheduler.alphas_cumprod = scheduler.alphas_cumprod.to(device)

In [ ]:
images = []
rewards = []
for batch in range(10):
    latents, next_latents, log_probs, timesteps = generate_batch(pretrained_model.module, scheduler, 10, device)
    rewards, _ = reward_function(next_latents[:, -1])
    images.append(next_latents[:, -1])
    rewards.append(rewards)

    del latents, next_latents, log_probs, timesteps
    torch.cuda.empty_cache()

images = torch.cat(images)
rewards = torch.cat(rewards)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sort images by reward (highest to lowest)
sorted_indices = torch.argsort(rewards, descending=True)
sorted_images = images[sorted_indices]
sorted_rewards = rewards[sorted_indices]

# Create a grid plot
num_images = sorted_images.shape[0]
cols = 5
rows = (num_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
if rows == 1:
    axes = axes.reshape(1, -1)

for i in range(num_images):
    row = i // cols
    col = i % cols
    
    # Process image for display
    image_processed = sorted_images[i].cpu().permute(1, 2, 0)
    image_processed = (image_processed + 1.0) * 127.5
    image_processed = image_processed.numpy().astype(np.uint8)
    
    axes[row, col].imshow(image_processed)
    axes[row, col].set_title(f"Reward: {sorted_rewards[i]:.3f}")
    axes[row, col].axis('off')

# Hide empty subplots
for i in range(num_images, rows * cols):
    row = i // cols
    col = i % cols
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()